# clearml-agent в Colab (T4)

Воркер очереди `colab-t4`: выполняет задачи, которые туда ставят — например, клоны
конфигураций сетки ёмкости ученика (`scripts/capacity_search.py`, ClearML GridSearch).

Перед запуском: **Среда выполнения → T4 GPU**, в **Secrets** — `CLEARML_API_ACCESS_KEY`
и `CLEARML_API_SECRET_KEY`.

Агент использует уже установленный Python Colab: venv не создаётся, torch не
переустанавливается, зависимости ставятся ячейкой ниже. Код задачи агент берёт из
репозитория (ветка в задаче), `--output_dir` задач указывает на Google Drive — после обрыва
сессии чекпоинты запусков остаются на Drive.

## 1. Google Drive и GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Нет GPU: смени среду выполнения на T4"

## 2. Код и зависимости

torch/torchvision в Colab уже стоят со сборкой под CUDA — их не переустанавливаем,
ставим только остальное из `requirements.txt`. Обучение идёт отдельным процессом
(`!python train.py`), поэтому предупреждения pip о версиях пакетов самого Colab не мешают.

In [ ]:
import os

REPO_URL = "https://github.com/esgon191/cifar100-distillation"
REPO_DIR = "/content/cifar100-distillation"
BRANCH = "master"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!grep -viE '^(torch|torchvision)==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. ClearML

Ключи берутся из Colab Secrets и передаются через переменные окружения — их наследует
процесс `train.py` (`Task.set_credentials` подействовал бы только на ядро ноутбука).

In [ ]:
from google.colab import userdata

for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
    os.environ[name] = userdata.get(name)
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

# Проверка авторизации без создания задачи
!python -c "from clearml.backend_api.session import Session; Session(); print('ClearML OK, files server:', Session.get_files_server_host())"

## 4. Агент

Ячейка блокирует ноутбук, пока агент работает; задачи выполняются по одной.
Ход обучения — в веб-интерфейсе ClearML. Если сессия оборвалась посреди задачи, её можно
перезапустить (Reset → Enqueue): каталог запуска `<output_dir>/<task_id>/run_0`
останется на Drive и обучение продолжится с чекпоинта.

In [ ]:
!pip install -q clearml-agent
import sys

os.environ["CLEARML_AGENT_SKIP_PIP_VENV_INSTALL"] = sys.executable
os.environ["CLEARML_AGENT_SKIP_PYTHON_ENV_INSTALL"] = "1"
os.makedirs("/content/drive/MyDrive/cifar100-distillation/out", exist_ok=True)
!clearml-agent daemon --queue colab-t4 --foreground